In [ ]:
import os
from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
from imitation.algorithms import bc
import gymnasium as gym
from imitation.data.types import Transitions
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from sdlarch_rl.utils.stf6_imitation import Re4CNN
from sdlarch_rl.utils.utils import get_last_index
import gc
from IPython import get_ipython
import cv2

gc.collect()
th.cuda.empty_cache()

rng = np.random.default_rng(0)

demo_path = 'demos/'

train_path = 'imitation/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 0 # 1e-3 # 0 # 1e-4
BATCH_SIZE= 128 # 64 # 128 # 32 # 64 # 128
NUMBER_OF_EPOCH= 45 # 35 # 30 # 27
EPOCH_PER_FILE= 2 # 1 # 2 # 1 # 3 # 2
IS_FLIPPED_OBS=True
MINI_BATCH=64
L2=1e-5
LEARNING_RATE= 1e-4# 2e-4
BUFFER_SIZE = 10 # 10 # 7 # 4 # 7
BUFFER_SIZE_ORIG = BUFFER_SIZE

action_space = gym.spaces.MultiBinary(7)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(4, 64, 64), # 4 frames 64x64
    dtype=np.uint8,
)


last_index = get_last_index(demo_path, "demos", "pt")
last_index_imitation = get_last_index(train_path, "bc_policy", "zip")

print("last_index: " + str(last_index))
print("Last training session saved: ", f"bc_policy{last_index_imitation}.zip")

files_index = np.arange(last_index + 1)
# files_index = np.arange(10, last_index + 1)

class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            **kwargs,
            features_extractor_class=StreetFighterCNN,
            # features_extractor_kwargs=dict(features_dim=256),
            features_extractor_kwargs=dict(features_dim=512),
            # net_arch=[dict(pi=[256, 256], vf=[256, 256])]
            net_arch=[dict(pi=[512, 512, 256], vf=[512, 512, 256])]
        )


policy = CustomActorCriticPolicy(
    observation_space=observation_space,
    action_space=action_space,
    lr_schedule=lambda _: th.finfo(th.float32).max,  # BC control the learning rate
)


th.serialization.add_safe_globals([Trajectory])


bc_trainer = bc.BC(
    observation_space=observation_space,
    action_space=action_space,
    rng=rng,
    ent_weight=ENT_WEIGHT,
    batch_size=BATCH_SIZE,
    policy=policy,
    optimizer_kwargs=dict(lr=LEARNING_RATE),
    l2_weight=L2,
    minibatch_size=MINI_BATCH,
)

def concat_transitions(list_of_transitions):
    return Transitions(
        obs=np.concatenate([t.obs for t in list_of_transitions]),
        acts=np.concatenate([t.acts for t in list_of_transitions]),
        next_obs=np.concatenate([t.next_obs for t in list_of_transitions]),
        dones=np.concatenate([t.dones for t in list_of_transitions]),
        infos=np.concatenate([t.infos for t in list_of_transitions]),
    )

def fix_action_format(acts):
    """
    Fix action shape
    """
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
        
        # if acts.dtype == np.float32 or acts.dtype == np.float64:
        #     acts = np.round(acts).astype(np.int8)
    
    return acts

def flip_obs(obs):
    # obs: (T+1, C, H, W)
    return np.flip(obs, axis=-1)

def flip_acts(acts):
    acts = acts.copy()
    acts[:, [2, 3]] = acts[:, [3, 2]]  # swap left/right
    return acts

epoch_count = 0

def fix_trajectories(trajectories):
    fixed_trajectories = []
            
    for traj in trajectories:
        # obs = np.array(traj.obs) if not isinstance(traj.obs, np.ndarray) else traj.obs
        obs = np.array(traj.obs)

        acts = fix_action_format(np.array(traj.acts, dtype=np.float32))

        # Case (T, 1, H, W, C)
        if obs.ndim == 5 and obs.shape[1] == 1:
            obs = obs[:, 0]  # remove dimension 1 → (T, H, W, C)
        
        # Case HWC → CHW
        if obs.ndim == 4 and obs.shape[-1] == 4:
            obs = obs.transpose(0, 3, 1, 2)  # (T, 4, 64, 64)

        print("obs shape", obs.shape)
        
        fixed_trajectories.append(
            Trajectory(
                obs=obs,
                acts=acts,
                infos=traj.infos,
                terminal=traj.terminal
            )
        )

        # FLIPPED (DATA AUGMENTATION)
        # 20 percent of prob
        if  np.random.rand() < 0.40 or IS_FLIPPED_OBS:
            flipped_obs = flip_obs(obs)
            flipped_acts = flip_acts(acts)

            fixed_trajectories.append(
                Trajectory(
                    obs=flipped_obs,
                    acts=flipped_acts,
                    infos=traj.infos,
                    terminal=traj.terminal,
                )
            )

    return fixed_trajectories

is_there_rest = False

for e in range(NUMBER_OF_EPOCH):
    np.random.shuffle(files_index)

    # reset buffer size
    BUFFER_SIZE = BUFFER_SIZE_ORIG

    epoch_count += 1

    print(f"\n--------------- Epoch: {epoch_count} ------------------\n")

    print("files_index: ", files_index)

    buffer = []

    buffer_files = []

    current_index = 0

    for i in files_index:
        is_there_rest = False
        trajectories = th.load(demo_path + f"demos{i}.pt", weights_only=False)

        # fix traj
        fixed_trajectories = fix_trajectories(trajectories)
            
        np.random.shuffle(fixed_trajectories)
        
        transitions = rollout.flatten_trajectories(fixed_trajectories)

        buffer.append(transitions)
        buffer_files.append(i)

        del transitions
        del fixed_trajectories
        del trajectories

        if len(buffer) == BUFFER_SIZE:
            merged = concat_transitions(buffer)

            print(f"Processing files: {buffer_files}")

            bc_trainer.set_demonstrations(merged)
            bc_trainer.train(n_epochs=EPOCH_PER_FILE)
            buffer.clear()
            buffer_files.clear()

            # save current epoch
            bc_trainer.policy.save(sub_train_path + f"bc_policy{epoch_count}.zip")
            is_there_rest = True

        if is_there_rest and len(files_index) - current_index < BUFFER_SIZE:
            print("current_index: ", current_index)
            rest_current = len(files_index) - current_index - 1
            print("Rest: ", rest_current)
            BUFFER_SIZE = rest_current

        current_index += 1


bc_trainer.policy.save(train_path + f"bc_policy{last_index_imitation + 1}.zip")

gc.collect()
bc_trainer._demonstrations = None
bc_trainer._demonstrations_tensor = None
del bc_trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 28
Last training session saved:  bc_policy10.zip


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:484: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(



--------------- Epoch: 1 ------------------

files_index:  [19 17 15  2 28 18 11 23 10  1 20 26 14 16  0  4 25 21  3  9 12 27 24  6
  8 22  7  5 13]
obs shape (648, 4, 64, 64)
obs shape (508, 4, 64, 64)
obs shape (758, 4, 64, 64)
obs shape (522, 4, 64, 64)
obs shape (938, 4, 64, 64)
obs shape (399, 4, 64, 64)
obs shape (1071, 4, 64, 64)
obs shape (923, 4, 64, 64)
obs shape (1015, 4, 64, 64)
obs shape (907, 4, 64, 64)
obs shape (711, 4, 64, 64)
obs shape (743, 4, 64, 64)
obs shape (529, 4, 64, 64)
obs shape (931, 4, 64, 64)
obs shape (580, 4, 64, 64)
obs shape (882, 4, 64, 64)
obs shape (1016, 4, 64, 64)
obs shape (685, 4, 64, 64)
obs shape (796, 4, 64, 64)
obs shape (568, 4, 64, 64)
obs shape (517, 4, 64, 64)
obs shape (553, 4, 64, 64)
obs shape (730, 4, 64, 64)
obs shape (561, 4, 64, 64)
obs shape (547, 4, 64, 64)
obs shape (791, 4, 64, 64)
obs shape (1178, 4, 64, 64)
obs shape (664, 4, 64, 64)
obs shape (674, 4, 64, 64)
obs shape (454, 4, 64, 64)
obs shape (446, 4, 64, 64)
obs shape

1batch [00:03,  3.72s/batch]

...

2995batch [00:42, 71.54batch/s]

--------------------------------
| batch_size        | 64       |
| bc/               |          |
|    batch          | 1500     |
|    ent_loss       | 0        |
|    entropy        | 0.0949   |
|    epoch          | 0        |
|    l2_loss        | 0.0544   |
|    l2_norm        | 5.44e+03 |
|    loss           | 0.146    |
|    neglogp        | 0.0919   |
|    prob_true_act  | 0.949    |
|    samples_so_far | 192128   |
--------------------------------


3762batch [00:53, 73.42batch/s]
4001batch [00:56, 71.68batch/s]

--------------------------------
| batch_size        | 64       |
| bc/               |          |
|    batch          | 2000     |
|    ent_loss       | 0        |
|    entropy        | 0.0529   |
|    epoch          | 1        |
|    l2_loss        | 0.0545   |
|    l2_norm        | 5.45e+03 |
|    loss           | 0.075    |
|    neglogp        | 0.0205   |
|    prob_true_act  | 0.983    |
|    samples_so_far | 256128   |
--------------------------------


4265batch [00:59, 74.50batch/s]